# Mooring Field Detection — Kaggle GPU Coastal Scan

YOLO detect on pre-fetched tiles (GPU T4). Code from GitHub; tiles/weights from attached dataset zip.

Local prep: `generate-candidates` → `fetch-scan` → `package-kaggle-scan` → upload zip → run this notebook → `import-scan` → `enrich-all --only-new`.

In [ ]:
# Cell 1 — sparse clone + install (avoid pulling training imagery into /kaggle/working)
import subprocess, sys, shutil, os
from pathlib import Path

REPO = Path("/kaggle/working/MooringFieldDetection")
URL = "https://github.com/IshanKasam/MooringFieldDetection.git"

if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(
    ["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", URL, str(REPO)],
    check=True,
)
subprocess.run(
    ["git", "-C", str(REPO), "sparse-checkout", "set", "src", "config", "pyproject.toml", "README.md"],
    check=True,
)
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics", "python-dotenv"],
    check=True,
)

import torch
from mooring_fields.runtime import cuda_available

print("cuda:", cuda_available(), torch.cuda.get_device_name(0) if cuda_available() else None)
assert cuda_available(), "Settings → Accelerator → GPU T4"

In [ ]:
# Cell 2 — materialize payload (folder or kaggle_scan_*.zip)
import json, os, sys, zipfile, shutil
from pathlib import Path

REPO = Path("/kaggle/working/MooringFieldDetection")
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

from mooring_fields.kaggle_scan import materialize_kaggle_scan_input

INPUT = Path("/kaggle/input")
payload = None
for p in sorted(INPUT.iterdir()) if INPUT.exists() else []:
    if (p / "candidates.kml").is_file() or list(p.rglob("candidates.kml")):
        payload = p
        break
    zips = sorted(p.glob("kaggle_scan_*.zip")) or sorted(p.glob("*.zip"))
    if zips:
        extract = Path("/kaggle/working/payload_extracted")
        if extract.exists():
            shutil.rmtree(extract)
        extract.mkdir(parents=True)
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(extract)
        payload = extract
        print("extracted", zips[0].name)
        break

assert payload is not None, "Attach mooring-scan-* dataset (zip or extracted layout)"
layout = materialize_kaggle_scan_input(payload)
print(json.dumps(layout, indent=2))
assert layout["png_count"] > 0 and layout["weights"]

In [ ]:
# Cell 3 — GPU detect (--skip-fetch)
import os, sys, zipfile, shutil
from pathlib import Path

REPO = Path("/kaggle/working/MooringFieldDetection")
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

from mooring_fields.cli import scan_cmd
from mooring_fields.kaggle_scan import materialize_kaggle_scan_input

extract = Path("/kaggle/working/payload_extracted")
if extract.exists() and (extract / "candidates.kml").is_file():
    payload = extract
else:
    INPUT = Path("/kaggle/input")
    payload = None
    for p in INPUT.iterdir():
        if (p / "candidates.kml").is_file() or list(p.rglob("candidates.kml")):
            payload = p
            break
        zips = sorted(p.glob("kaggle_scan_*.zip")) or sorted(p.glob("*.zip"))
        if zips:
            if extract.exists():
                shutil.rmtree(extract)
            extract.mkdir(parents=True)
            with zipfile.ZipFile(zips[0]) as zf:
                zf.extractall(extract)
            payload = extract
            break
    assert payload is not None

layout = materialize_kaggle_scan_input(payload)
out_dir = Path("/kaggle/working/scan_out")
out_dir.mkdir(parents=True, exist_ok=True)

scan_cmd([
    "--kml", layout["kml"],
    "--skip-fetch",
    "--imagery-dir", layout["imagery_dir"],
    "--weights", layout["weights"],
    "--db", str(out_dir / "mooring_fields.db"),
    "--output-dir", str(out_dir),
])

db = out_dir / "mooring_fields.db"
assert db.is_file()
print("DOWNLOAD →", db)
shutil.rmtree(REPO, ignore_errors=True)
shutil.rmtree(extract, ignore_errors=True)